<a href="https://colab.research.google.com/github/iras-mpark/MLA1020/blob/main/week12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from typing import Callable
import numpy as np

class ProbTable:
    """
    Represents an arbitrary probability table: could be a local conditional
    distribution, a marginal distribution, or a conditional distribution.
    description:
      A         B=1          | C          D=1
      gen-vars  gen-vals       cond-vars  cond-vals
    data could be:
    - tensor corresponding to the probabilities (axes are cond-vars + gen-vars)
    - function mapping cond-vars + gen-vars assignments to probabilities
    shape: only needed if data is a function, to determine the shape of the tensor
    """
    def __init__(self, description: str, data: np.ndarray | Callable, shape: tuple[int] | None = None):
        if isinstance(data, Callable):
            # Build up the probabilities
            self.probs = np.empty(shape)
            def recurse(assignment: list[int]):
                if len(assignment) == len(shape):
                    self.probs[tuple(assignment)] = data(*assignment)
                else:
                    for i in range(shape[len(assignment)]):
                        recurse(assignment + [i])
            recurse([])
        else:
            self.probs = np.array(data)
        # Parse the description into variables and values for both the generation and conditioning side
        self.cond_vars = []
        self.gen_vars = []
        self.cond_vals = []
        self.gen_vals = []
        items = description.split(" ")
        on_conditioning_side = False
        for item in items:
            if item == "|":
                on_conditioning_side = True
            elif on_conditioning_side:
                if "=" in item:
                    self.cond_vals.append(item)
                else:
                    self.cond_vars.append(item)
            else:
                if "=" in item:
                    self.gen_vals.append(item)
                else:
                    self.gen_vars.append(item)
        assert len(self.cond_vars) + len(self.gen_vars) == len(self.probs.shape), [self.cond_vars, self.gen_vars, self.probs.shape]
    @property
    def p(self):
        return self.probs
    def asdict(self):
        """
        Returns a nice matrix representation of the probability table.
        a  b  P(A = a, B = b)
        0  0  0.1
        0  1  0.2
        1  0  0.3
        1  1  0.4
        """
        vars = self.cond_vars + self.gen_vars
        # Description string (e.g., "P(A = a, B = b)")
        output = []
        output.append(", ".join([f"{var}={var.lower()}" for var in self.gen_vars] + self.gen_vals))
        if len(self.cond_vars) > 0 or len(self.cond_vals) > 0:
            output.append("|")
            output.append(", ".join([f"{var}={var.lower()}" for var in self.cond_vars] + self.cond_vals))
        prob_str = "P(" + " ".join(output) + ")"
        rows = []
        rows.append([var.lower() for var in vars] + [prob_str])
        def recurse(assignment: list):
            if len(assignment) == len(vars):
                rows.append(assignment + [self.probs[tuple(assignment)]])
            else:
                for value in range(self.probs.shape[len(assignment)]):
                    recurse(assignment + [value])
        recurse([])
        return np.array(rows)

In [7]:
from einops import einsum

P_SR = ProbTable("S R", [[0.22, 0.08], [0.6, 0.1]])
print("P(S, R) =\n", P_SR.p)

P_S = ProbTable("S", einsum(P_SR.p, "s r -> s"))
print("P(S) =\n", P_S.p)

P(S, R) =
 [[0.22 0.08]
 [0.6  0.1 ]]
P(S) =
 [0.3 0.7]
